# 02 — Radiomics Extraction Demo

Interactive demonstration of radiomic feature extraction for a single patient.
Requires preprocessed NIfTI images and tumor masks.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
from utils import load_config, load_nifti, image_to_array

cfg = load_config('../config/config.yaml')
%matplotlib inline

In [ ]:
# Select a patient for demonstration
proc_root = Path('../data/processed')
region_root = Path('../data/processed/regions')

patients = sorted([d.name for d in proc_root.iterdir()
                   if d.is_dir() and d.name != 'regions'])
print(f'Available patients: {patients[:5]}  (showing first 5)')

# Change this to any available patient ID
PATIENT_ID = patients[0] if patients else 'GBM-001'
TIMEPOINT  = 'baseline'
print(f'Using: {PATIENT_ID} / {TIMEPOINT}')

In [ ]:
# Load images
pt_dir = proc_root / PATIENT_ID / TIMEPOINT
reg_dir = region_root / PATIENT_ID / TIMEPOINT

t1gd_path  = pt_dir / 't1gd.nii.gz'
flair_path = pt_dir / 'flair.nii.gz'
mask_path  = reg_dir / 'tumor_mask.nii.gz'
ring_path  = reg_dir / 'ring_5mm.nii.gz'

imgs = {}
for name, path in [('T1Gd', t1gd_path), ('FLAIR', flair_path),
                    ('Tumor Mask', mask_path), ('Ring 5mm', ring_path)]:
    if path.exists():
        imgs[name] = image_to_array(load_nifti(path))
        print(f'{name}: {imgs[name].shape}, spacing={load_nifti(path).GetSpacing()}')
    else:
        print(f'{name}: NOT FOUND at {path}')

In [ ]:
# Visualise
if 'Tumor Mask' in imgs:
    z_idx = np.where(imgs['Tumor Mask'].sum(axis=(1,2)) > 0)[0]
    z = int(z_idx[len(z_idx)//2]) if len(z_idx) > 0 else imgs['T1Gd'].shape[0]//2
else:
    z = imgs['T1Gd'].shape[0]//2 if 'T1Gd' in imgs else 0

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
panel_data = [
    ('T1Gd', None, 'T1Gd'),
    ('T1Gd', 'Tumor Mask', 'T1Gd + Tumor'),
    ('FLAIR', 'Tumor Mask', 'FLAIR + Tumor'),
    ('T1Gd', 'Ring 5mm', 'T1Gd + Ring 5mm'),
]
for ax, (base_key, overlay_key, title) in zip(axes, panel_data):
    base = imgs.get(base_key)
    if base is not None:
        ax.imshow(base[z], cmap='gray', origin='lower')
    if overlay_key and overlay_key in imgs:
        mask = imgs[overlay_key][z]
        masked = np.ma.masked_where(mask == 0, mask)
        ax.imshow(masked, cmap='Reds' if 'Mask' in overlay_key else 'Blues',
                  alpha=0.45, origin='lower')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig.suptitle(f'{PATIENT_ID} — Axial slice {z}', fontsize=12)
plt.tight_layout()
plt.savefig(f'../results/figures/example_patient_{PATIENT_ID}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Run PyRadiomics on a single region
import logging, warnings
logging.getLogger('radiomics').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

if t1gd_path.exists() and mask_path.exists():
    from radiomics import featureextractor
    extractor = featureextractor.RadiomicsFeatureExtractor('../config/pyradiomics_params.yaml')
    result = extractor.execute(str(t1gd_path), str(mask_path))
    
    import pandas as pd
    feat = {k: float(v) for k, v in result.items() if not k.startswith('diagnostics_')}
    df_feat = pd.Series(feat)
    print(f'\nExtracted {len(df_feat)} features from T1Gd / Tumor Mask')
    
    # Show breakdown by feature class
    for cls in ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm']:
        n = sum(1 for k in feat if f'original_{cls}' in k)
        print(f'  {cls:<15}: {n} features')
    print(f'  LoG + Wavelet : {sum(1 for k in feat if "LoG" in k or "wavelet" in k)} features')
else:
    print('Images not available for this patient. Run preprocess_mri.py first.')